# Matplotlib - Visualisation de données

Notebook de l'article [Matplotlib - Visualisation de données](https://moncoachdata.com/blog/matplotlib-visualisation-de-donnees/) (MonCoachData). Exécute les cellules dans l'ordre.

La visualisation de données est une étape clé pour comprendre un jeu de données et en tirer des conclusions. Tu peux toujours inspecter les données ligne par ligne, cellule par cellule, mais c'est fastidieux et cela ne fait pas ressortir la vue d'ensemble.

Les visuels, eux, présentent les données sous une forme compréhensible d'un coup d'œil, et ils retiennent l'attention du public.

Dans cet article sur la **visualisation de données avec Matplotlib**, tu vas apprendre à créer des graphiques linéaires, des diagrammes en barres et des diagrammes de dispersion, à les personnaliser, et surtout à en tirer des conclusions. Matplotlib est la bibliothèque de base de la visualisation en Python : elle propose de nombreux types de graphiques et une personnalisation très poussée (étiquettes, titres, couleurs, tailles de police…). Plus de détails sur [matplotlib.org](https://matplotlib.org/).

Pour s'entraîner, on utilise la **densité de population** des pays du monde (le nombre d'habitants par km²) de 1960 à 2015.

**En bref :** Matplotlib est la bibliothèque de base de la visualisation de données en Python. Cet article l'utilise sur la densité de population de 214 pays de 1960 à 2015 pour créer des graphiques linéaires, des diagrammes en barres et des diagrammes de dispersion, et les personnaliser (titres, étiquettes, légendes, échelles). La règle à retenir : une courbe pour suivre une évolution, des barres pour comparer des catégories (en gardant les plus importantes), un nuage de points pour étudier la relation entre deux variables.

## Comment installer et importer Matplotlib et Pandas ?

Si Matplotlib et Pandas ne sont pas installés, ajoute-les avec pip :

Comme toujours, on commence par importer les bibliothèques. **Pandas** sert à préparer les données, et le module **pyplot** de **Matplotlib** à tracer les graphiques :

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

Dans un notebook Jupyter ancien, on ajoutait aussi la commande `%matplotlib inline` pour afficher les graphiques dans le notebook. Les versions récentes de Jupyter n'en ont plus besoin. Dans un script Python, termine chaque graphique par `plt.show()`.

## Comment construire le jeu de données de densité de population ?

On part de deux sources publiques : la population de chaque pays (jeu `datasets/population`) et sa superficie (jeu `mledoze/countries`), dont une copie figée est hébergée sur le GitHub de MonCoachData. La densité de population, c'est simplement la population divisée par la superficie.

On fusionne les deux tables avec `merge()` sur le code du pays. Le fichier de population contient aussi des regroupements (« World », « Euro area »…) : comme ils n'ont pas de superficie, la fusion les élimine. On garde les années de 1960 à 2015, puis on transforme le tableau avec `pivot()` pour avoir un pays par ligne et une année par colonne :

In [ ]:
url_population = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/matplotlib-visualisation-de-donnees/population.csv"
url_pays = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/matplotlib-visualisation-de-donnees/countries.json"

population = pd.read_csv(url_population)
pays = pd.read_json(url_pays)[["cca3", "area"]]

donnees = population.merge(pays, left_on="Country Code", right_on="cca3")
donnees = donnees[donnees["Year"] <= 2015].copy()
donnees["densite"] = donnees["Value"] / donnees["area"]

densite = donnees.pivot(index="Country Name", columns="Year", values="densite")
print(densite.shape)
print(densite.iloc[:3, [0, -1]].round(1))

Chaque ligne est un pays et chaque colonne une année. Vérifions les valeurs manquantes :

In [ ]:
print(densite.isnull().sum().sum())

densite = densite.dropna()
print(densite.shape)

Quelques valeurs sont manquantes : `dropna()` supprime les pays concernés. Il reste 214 pays, avec un historique complet de 1960 à 2015.

Pour la suite, on garde aussi la population et la superficie de chaque pays en 2015 :

In [ ]:
infos = (donnees[donnees["Year"] == 2015]
         .set_index("Country Name")[["Value", "area"]]
         .rename(columns={"Value": "population", "area": "superficie"})
         .loc[densite.index])

Les données sont prêtes : on peut les visualiser.

## Comment visualiser les données avec Matplotlib ?

On utilise maintenant Matplotlib pour créer nos graphiques, et on en tire des conclusions.

### Comment tracer un graphique linéaire ?

Pour tracer un graphique linéaire avec Matplotlib, appelle `plot()` avec deux séries : les valeurs de l'axe des x et celles de l'axe des y. On commence par la densité de population d'**Aruba** au fil des années, avec un graphique linéaire : les années sur l'axe des x, la densité sur l'axe des y.

On récupère les années avec `densite.columns` et la densité d'Aruba avec `densite.loc["Aruba"]`. Il suffit ensuite d'appeler la fonction `plot()` avec ces deux séries :

In [ ]:
x = densite.columns
y = densite.loc["Aruba"]

plt.plot(x, y)
plt.show()

Félicitations, tu as créé ton premier graphique !

Il fait le travail, mais il est difficile à interpréter : pas de titre, pas d'étiquettes sur les axes. C'est là que la personnalisation de Matplotlib devient utile.

Pour mieux contrôler le résultat, on utilise l'approche recommandée : `plt.subplots()` crée une **figure** (`fig`) et un **axe** (`ax`), puis on règle chaque détail avec des méthodes de l'axe :

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(x, y, linewidth=3)
ax.set_title("Densité de population d'Aruba, de 1960 à 2015")
ax.set_xlabel("Année")
ax.set_ylabel("Densité de population (habitants par km²)")
ax.grid(alpha=0.3)

plt.show()

- `figsize` définit la taille de la figure, en pouces ;
- `linewidth` règle l'épaisseur de la ligne ;
- `set_title()`, `set_xlabel()` et `set_ylabel()` ajoutent le titre et les étiquettes des axes ;
- `grid()` ajoute une grille discrète.

Le graphique est maintenant beaucoup plus clair, avec quelques lignes seulement.

**Ce qu'on en tire :** la densité d'Aruba est restée presque stable jusqu'en 1990 (de 305 à 349 habitants par km²). Elle a ensuite fortement augmenté dans les années 1990 (jusqu'à 503 en 2000), puis a continué à croître plus lentement jusqu'à 599 en 2015.

### Comment comparer plusieurs courbes sur un même graphique ?

Les graphiques linéaires servent aussi à comparer la tendance de plusieurs pays. Comparons 5 pays : on trace une courbe par pays dans la même figure. Chaque courbe reçoit un `label`, et `legend()` affiche la légende qui associe une couleur à chaque pays. Matplotlib choisit automatiquement des couleurs différentes :

In [ ]:
pays_choisis = ["Aruba", "Andorra", "Afghanistan", "Angola", "Albania"]

fig, ax = plt.subplots(figsize=(10, 5))

for nom in pays_choisis:
    ax.plot(densite.columns, densite.loc[nom], label=nom, linewidth=3)

ax.set_title("Comparaison de la densité de population de 5 pays")
ax.set_xlabel("Année")
ax.set_ylabel("Densité de population (habitants par km²)")
ax.legend()

plt.show()

**Ce qu'on en tire :** avec toutes les courbes sur une seule figure, on voit tout de suite qu'Aruba a toujours eu une densité bien supérieure à celle des 4 autres pays (599 habitants par km² en 2015, contre 154 pour Andorre et 23 pour l'Angola). Entre 1960 et 2015, la densité augmente nettement pour Aruba, Andorre (qui recule toutefois après 2010), l'Afghanistan et l'Angola. L'Albanie fait exception : sa densité recule depuis 1990 (114 habitants par km² en 1990, 95 en 2015).

### Comment créer un diagramme en barres lisible ?

On crée un diagramme en barres avec la méthode `bar()`. Essayons d'abord de tracer la densité de population de **tous** les pays pour l'année 2015 :

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(densite.index, densite[2015])
ax.set_title("Densité de population de tous les pays en 2015")
ax.tick_params(axis="x", labelrotation=90)

plt.show()

Ce graphique essaie de montrer trop d'informations à la fois : les étiquettes de l'axe des x se chevauchent et deux ou trois barres écrasent toutes les autres. Il est inutilisable.

Il ne suffit pas de visualiser un jeu de données : il faut en visualiser intelligemment les parties importantes. Ici, on trie les pays par densité décroissante avec `sort_values()` et on garde les 10 premiers avec `head(10)` :

In [ ]:
top10 = densite[2015].sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(top10.index, top10.values, color="tab:blue")
ax.set_title("Les 10 pays les plus denses en 2015")
ax.set_ylabel("Densité de population (habitants par km²)")
ax.tick_params(axis="x", labelrotation=45)
plt.setp(ax.get_xticklabels(), ha="right")

fig.tight_layout()
plt.show()

L'information est enfin claire : les barres sont bien espacées, les noms lisibles. On a aussi pris une seule couleur : ici, les couleurs ne portent aucune information, et en mettre plusieurs ne ferait que distraire. `tight_layout()` ajuste les marges pour que rien ne soit coupé.

**Ce qu'on en tire :** Macao (21 430 habitants par km²) et Monaco (18 197) ont de loin la plus forte densité de population en 2015, devant Singapour (7 796) et Hong Kong (6 604).

### Comment créer un diagramme de dispersion ?

Un diagramme de dispersion (*scatter plot*) affiche chaque donnée sous forme de point, pour montrer la relation entre deux variables. On peut aussi y faire varier la couleur (`c`) ou la taille (`s`) des points selon une troisième variable.

Ici, on place chaque pays selon sa superficie (axe x) et sa population (axe y), et on colore le point selon sa densité de population. Comme les valeurs vont de quelques milliers à plusieurs milliards, on utilise une **échelle logarithmique** sur les deux axes, et `LogNorm()` pour l'échelle de couleurs :

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

points = ax.scatter(infos["superficie"], infos["population"],
                    c=densite[2015], s=40, cmap="viridis", norm=LogNorm())
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Superficie, population et densité des pays en 2015")
ax.set_xlabel("Superficie (km², échelle logarithmique)")
ax.set_ylabel("Population (échelle logarithmique)")

for nom in ["Monaco", "Macao SAR, China", "Greenland", "Bangladesh", "China"]:
    ax.annotate(nom, (infos.loc[nom, "superficie"], infos.loc[nom, "population"]),
                xytext=(6, 6), textcoords="offset points")

fig.colorbar(points, label="Densité de population (habitants par km²)")
plt.show()

Les pays alignés sur une diagonale ont la même densité : ceux qui sont en haut à gauche (petite superficie, grande population) sont les plus denses, ceux d'en bas à droite les moins denses. `annotate()` permet d'écrire le nom de quelques pays repères.

**Ce qu'on en tire :** Monaco et Macao, minuscules mais très peuplés par rapport à leur taille, sont tout en haut à gauche. Le Groenland, immense et presque vide, est en bas à droite : c'est le pays dont la densité moyenne sur la période est la plus faible (0,03 habitant par km² en 2015).

## Analyse approfondie

Allons plus loin dans l'analyse du jeu de données.

### Comment l'écart de densité évolue-t-il ?

On compare la densité maximale, médiane et minimale de chaque année. Comme les écarts sont énormes, on utilise une échelle logarithmique (`set_yscale("log")`) sur l'axe des y :

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(densite.columns, densite.max(), label="Maximum", linewidth=3)
ax.plot(densite.columns, densite.median(), label="Médiane", linewidth=3)
ax.plot(densite.columns, densite.min(), label="Minimum", linewidth=3)

ax.set_yscale("log")
ax.set_title("Densité de population maximale, médiane et minimale par année")
ax.set_xlabel("Année")
ax.set_ylabel("Densité (habitants par km², échelle logarithmique)")
ax.legend(loc="center right")

plt.show()

**Ce qu'on en tire :** les trois courbes montent. Le pays le plus dense double sa densité (de 10 796 à 21 430 habitants par km²), la densité médiane est multipliée par près de 3 (de 31,5 à 84,7) et le pays le moins dense, le Groenland, reste à moins de 0,03 habitant par km². L'écart entre les pays les plus et les moins denses s'est donc encore creusé.

### La densité de population mesure-t-elle la taille d'un pays ?

La densité de population est-elle une bonne mesure de la taille d'un pays ? Pour le vérifier, on met côte à côte la population et la densité des 12 pays les plus denses. On crée une figure à deux axes avec `plt.subplots(1, 2)` : une ligne, deux colonnes.

In [ ]:
top12 = densite[2015].sort_values(ascending=False).head(12).index

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(top12, infos.loc[top12, "population"] / 1e6, color="tab:red")
ax1.set_title("Population en 2015 (millions d'habitants)")

ax2.bar(top12, densite.loc[top12, 2015], color="tab:blue")
ax2.set_title("Densité de population en 2015 (habitants par km²)")

for ax in (ax1, ax2):
    ax.tick_params(axis="x", labelrotation=90)

fig.tight_layout()
plt.show()

**Ce qu'on en tire :** la densité n'est pas un bon indicateur de la population. Les pays les plus denses sont presque tous minuscules : leur population est faible. Le Bangladesh, onzième de ce classement avec 1 080 habitants par km², compte 159 millions d'habitants, contre 5,5 pour Singapour, pourtant sept fois plus dense.

Tu peux enregistrer n'importe quelle figure avec `fig.savefig("graphique.png", dpi=150)`.